# 🤖 AIOS Google Colab LLM Coding Server (Qwen2.5-Coder-7B-Instruct)

Этот ноутбук запускает **100% бесплатный сервер vLLM / Ollama** с топовой нейросетью для написания кода `Qwen2.5-Coder-7B-Instruct` на GPU Google Colab и пробрасывает публичный туннель в AIOS.

### 🚀 Порядок действий:
1. Нажмите **`Среда выполнения` ➔ `Сменить тип среды выполнения` ➔ `T4 GPU`**.
2. Выполните Ячейку 1 для установки зависомостей.
3. Выполните Ячейку 2 для запуска модели на GPU.
4. Скопируйте ссылку туннеля из Ячейки 3 и вставьте команду в Telegram или консоль AIOS.

In [ ]:
# === ЯЧЕЙКА 1: Установка vLLM и Cloudflared ===
!pip install vllm cloudflared -q
print('✅ Зависимости успешно установлены!')

In [ ]:
# === ЯЧЕЙКА 2: Запуск vLLM OpenAI API сервера на GPU ===
import subprocess, time

print('🚀 Загрузка кодинг-модели Qwen2.5-Coder-7B-Instruct на GPU...')
vllm_cmd = 'python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-Coder-7B-Instruct --port 8000 --trust-remote-code --max-model-len 8192'
vllm_proc = subprocess.Popen(vllm_cmd, shell=True)
print('⏳ Ожидание полной загрузки модели в память GPU (~45 секунд)...')
time.sleep(45)
print('✅ vLLM OpenAI API Server запущен на порту 8000!')

In [ ]:
# === ЯЧЕЙКА 3: Создание публичного туннеля для AIOS ===
import subprocess, re, time

print('📡 Запуск туннеля Cloudflare...')
tunnel = subprocess.Popen('cloudflared tunnel --url http://localhost:8000', shell=True, stderr=subprocess.PIPE, text=True)

time.sleep(5)
for line in iter(tunnel.stderr.readline, ''):
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0) + '/v1'
            print('\n🎉 ========================================================')
            print('🔗 ПУБЛИЧНЫЙ URL ВАШЕЙ КОДИНГ-НЕЙРОСЕТИ В GOOGLE COLAB:')
            print(f'   {tunnel_url}')
            print('========================================================\n')
            print('👉 Выполните эту команду на сервере AIOS для регистрации:')
            print(f'python scripts/register_colab_llm.py {tunnel_url}')
            print('========================================================\n')
            break